# functional-module-wrap — worked example 3: A Conv2d wrapper around F.conv2d

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `functional-module-wrap`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The real payoff of the pattern is parameterized layers: the module *owns* the kernel as an `nn.Parameter` and `forward` delegates the heavy lifting to `F.conv2d(x, self.weight, ...)`. This is exactly how `nn.Conv2d` is implemented — a parameter container plus a one-line functional call.

## Worked solution

We implement a minimal `MyConv2d` that mirrors `nn.Conv2d` (no bias, for brevity).

1. `__init__` builds the weight tensor of shape `(out_channels, in_channels, k, k)` and registers it as `self.weight = nn.Parameter(...)`. Registering as a parameter means it shows up in `parameters()` and gets gradients.
2. We store `stride` and `padding` as plain attributes — they are config.
3. `forward(x)` returns `F.conv2d(x, self.weight, stride=self.stride, padding=self.padding)`. The module holds the kernel; the functional does the convolution.
4. We verify our output matches a real `nn.Conv2d` after copying our weight into it — proving the wrap is numerically identical — and that our module reports exactly one parameter (the weight).

In [ ]:
import torch as t
import torch.nn as nn
import torch.nn.functional as F

t.manual_seed(2)

class MyConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, k, stride=1, padding=0):
        super().__init__()
        self.stride = stride
        self.padding = padding
        self.weight = nn.Parameter(t.randn(out_ch, in_ch, k, k))

    def forward(self, x):
        return F.conv2d(x, self.weight, stride=self.stride, padding=self.padding)

x = t.randn(1, 3, 8, 8)
m = MyConv2d(3, 4, 3, stride=1, padding=1)
ref = nn.Conv2d(3, 4, 3, stride=1, padding=1, bias=False)
with t.no_grad():
    ref.weight.copy_(m.weight)
print('matches nn.Conv2d:', bool(t.allclose(m(x), ref(x), atol=1e-5)))
print('num params:', sum(1 for _ in m.parameters()))